# Week 5 — Track A: RAG by hand

**SWE 485 · Advanced Software Engineering with AI · King Saud University**

You are going to build the retrieve → augment → generate loop yourself, over the
`minibank/` corpus, with every similarity score printed. Then you are going to
break it five times on purpose.

The one rule: **the cosine calculation must be yours.** Do not import a
similarity helper. The whole point of this lab is that "the tool found the
relevant code" reduces to a dot product and two magnitudes, and you only get
that by writing the line.

---


In [1]:
from google.colab import files

uploaded = files.upload()

Saving swe485-week05-rag-seed.zip to swe485-week05-rag-seed.zip


In [2]:
import zipfile
import os

zip_file = "swe485-week05-rag-seed.zip"  # change to your zip name

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall("dataset")

os.listdir("dataset")

['swe485-week05-rag-seed']

## Setup

Your API key must be stored as the Colab Secret `GEMINI_API_KEY`, not pasted
into a cell. Get one at [aistudio.google.com/apikey](https://aistudio.google.com/apikey).


In [10]:
%pip install -U -q "google-genai" numpy

In [12]:
import numpy as np
from google import genai
from google.genai import types

# In Colab: clone the seed repo first, then run this notebook from inside it.
#   !git clone <your-fork-url> && cd swe485-week05-rag-seed
import config

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:
    import os
    GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]

client = genai.Client(api_key=GEMINI_API_KEY)

print("embedding model :", config.EMBED_MODEL)
print("chat model      :", config.CHAT_MODEL)
print("offline mode    :", config.OFFLINE)

embedding model : gemini-embedding-001
chat model      : gemini-3.6-flash
offline mode    : False


---

## A1 — Chunk

Chunking decides the **unit of retrieval**. Everything downstream inherits the
decision you make here, which is why the study material calls it the most
consequential design choice in the pipeline.

Read the next cell before you run it. You will be asked about it.


In [13]:
import re, fnmatch, pathlib

def tokenize(text):
    """An approximation of the model's tokenizer: words and standalone symbols.

    It is not the real tokenizer, and it does not need to be. What it must be
    is DETERMINISTIC, so that every team's chunk boundaries land in the same
    place and the examiner can predict what you will see.
    """
    return re.findall(r"\w+|[^\w\s]", text)


def load_ragignore(root):
    """Read the exclusion list. Open the file. Actually read it."""
    patterns, f = [], root / ".ragignore"
    if f.exists():
        for line in f.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#"):
                patterns.append(line)
    return patterns


def is_ignored(rel, patterns):
    return any(
        fnmatch.fnmatch(rel, p) or fnmatch.fnmatch(rel, p.rstrip("/") + "/*")
        for p in patterns
    )


def chunk_file(path, rel, chunk_tokens, overlap_tokens):
    """Fixed-size chunking, aligned to line boundaries.

    Lines are accumulated until the token budget is spent. The next chunk backs
    up by up to `overlap_tokens` so that a cut point is not a total loss.
    """
    lines = path.read_text().splitlines()
    counts = [len(tokenize(l)) for l in lines]
    chunks, start = [], 0
    while start < len(lines):
        total, end = 0, start
        while end < len(lines) and total + counts[end] <= chunk_tokens:
            total += counts[end]
            end += 1
        if end == start:            # one line longer than the whole budget
            end = start + 1
        chunks.append({
            "id": f"{rel}#L{start+1}-L{end}",
            "path": rel, "start": start + 1, "end": end,
            "text": "\n".join(lines[start:end]), "tokens": total,
        })
        if end >= len(lines):
            break
        back, carried = end, 0
        while back > start + 1 and carried + counts[back-1] <= overlap_tokens:
            back -= 1
            carried += counts[back]
        start = back
    return chunks


def build_chunks(corpus_dir, chunk_tokens, overlap_tokens, suffixes):
    root = pathlib.Path(corpus_dir)
    patterns = load_ragignore(root)
    chunks, skipped = [], []
    for p in sorted(root.rglob("*")):
        if not p.is_file() or p.suffix not in suffixes:
            continue
        rel = str(p.relative_to(root))
        if is_ignored(rel, patterns) or p.name == "__init__.py":
            skipped.append(rel)
            continue
        chunks += chunk_file(p, rel, chunk_tokens, overlap_tokens)
    return chunks, skipped

In [14]:
chunks, skipped = build_chunks(
    config.CORPUS_DIR, config.CHUNK_TOKENS, config.OVERLAP_TOKENS,
    config.INDEXABLE_SUFFIXES,
)

print(f"{len(chunks)} chunks indexed, {len(skipped)} files skipped\n")
for ch in chunks:
    print(f"  {ch['id']:36s} {ch['tokens']:4d} tokens")

print("\nNOT INDEXED:")
for s in skipped:
    print("  ", s)

10 chunks indexed, 4 files skipped

  auth/login.py#L1-L54                  386 tokens
  auth/password.py#L1-L34               200 tokens
  auth/tokens.py#L1-L68                 487 tokens
  auth/tokens.py#L61-L95                278 tokens
  billing/charges.py#L1-L60             434 tokens
  billing/invoice.py#L1-L22             166 tokens
  billing/refund.py#L1-L39              294 tokens
  config/settings.py#L1-L29             249 tokens
  docs/ADR-002-token-expiry.md#L1-L44   457 tokens
  docs/README.md#L1-L33                 278 tokens

NOT INDEXED:
   auth/__init__.py
   auth/mfa.py
   billing/__init__.py
   config/__init__.py


**Stop and look at that output before going on.**

1. Which file produced **two** chunks instead of one, and where did the cut land?
   Open it and read the lines either side of the boundary. What ended up
   separated from what?
2. Which file is in the "NOT INDEXED" list for a reason that is not `__init__.py`?
   Find out why. The answer is in a file you have not opened yet.

Those two observations are failure modes 1 and 2, and you have just found both
of them before running a single query.


---

## A2 — Embed

Retrieval embeddings are **asymmetric**. A question and the passage that answers
it are not the same kind of text, and the API makes you say which is which:
`RETRIEVAL_DOCUMENT` for chunks, `CODE_RETRIEVAL_QUERY` for questions about code.

**Normalization.** `gemini-embedding-001` returns un-normalized vectors at any
dimensionality other than 3072. We ask for 768, so we must normalize ourselves.
This is section 3's transfer question in production: normalize at index time and
cosine collapses to a plain dot product.


In [15]:
def normalize(v):
    v = np.asarray(v, dtype=np.float32)
    return v / np.linalg.norm(v)


def embed_documents(texts):
    result = client.models.embed_content(
        model=config.EMBED_MODEL,
        contents=[config.prepare_document(t) for t in texts],
        config=types.EmbedContentConfig(
            task_type=config.DOC_TASK_TYPE,
            output_dimensionality=config.EMBED_DIM,
        ),
    )
    vecs = np.stack([normalize(e.values) for e in result.embeddings])
    return vecs


if config.OFFLINE:
    cached = np.load("embeddings.npz", allow_pickle=True)
    doc_vectors = cached["vectors"]
    assert list(cached["ids"]) == [c["id"] for c in chunks], (
        "embeddings.npz does not match the current corpus or chunk settings. "
        "Re-run tools/build_embeddings.py."
    )
    print("loaded", doc_vectors.shape, "from embeddings.npz")
else:
    doc_vectors = embed_documents([c["text"] for c in chunks])
    print("embedded", doc_vectors.shape)

# Sanity check: normalized vectors have length 1.
print("norm of first vector:", float(np.linalg.norm(doc_vectors[0])))

embedded (10, 768)
norm of first vector: 1.0


---

## A3 — Retrieve  *(the graded step)*

Top-k is a k-nearest-neighbour search: embed the query, score every stored chunk
by cosine similarity, sort, take the top *k*. That is all it has ever been.

Because the document vectors are already normalized, `cosine()` below reduces to
`a @ b`. Write it out in full anyway the first time — you need to be able to say
what each term is.


In [20]:
def cosine(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


def embed_query(query):
    result = client.models.embed_content(
        model=config.EMBED_MODEL,
        contents=[config.prepare_query(query)],
        config=types.EmbedContentConfig(
            task_type=config.QUERY_TASK_TYPE,
            output_dimensionality=config.EMBED_DIM,
        ),
    )
    return normalize(result.embeddings[0].values)


def retrieve(query, k=None, show_all=True):
    """Score every chunk, print the ranking, return the top k."""
    k = config.K if k is None else k
    qv = embed_query(query)
    scored = sorted(
        ((cosine(qv, doc_vectors[i]), chunks[i]) for i in range(len(chunks))),
        key=lambda pair: pair[0],
        reverse=True,
    )

    print(f"QUERY: {query}\n")
    shown = scored if show_all else scored[:k]
    for rank, (score, ch) in enumerate(shown, 1):
        marker = "<-- retrieved" if rank <= k else ""
        print(f"  {rank:2d}. {score:.4f}  {ch['id']:36s} {marker}")

    gap = scored[0][0] - scored[1][0]
    print(f"\n  margin between rank 1 and rank 2: {gap:.4f}")
    return [ch for _, ch in scored[:k]]

### Q0 — the control

This is the worked trace from the lecture, run for real. In section 3 you
hand-computed that a login query scored ≈0.99 against an auth chunk and ≈0.44
against a billing chunk, using two-dimensional toy vectors. These are 768
dimensions. Watch the same thing happen.


In [21]:
top = retrieve("How does login work?")

QUERY: How does login work?

   1. 0.7817  auth/login.py#L1-L54                 <-- retrieved
   2. 0.7225  auth/tokens.py#L1-L68                <-- retrieved
   3. 0.7189  docs/README.md#L1-L33                <-- retrieved
   4. 0.7049  auth/password.py#L1-L34              <-- retrieved
   5. 0.6932  config/settings.py#L1-L29            
   6. 0.6893  docs/ADR-002-token-expiry.md#L1-L44  
   7. 0.6763  auth/tokens.py#L61-L95               
   8. 0.6680  billing/charges.py#L1-L60            
   9. 0.6268  billing/invoice.py#L1-L22            
  10. 0.6079  billing/refund.py#L1-L39             

  margin between rank 1 and rank 2: 0.0591


---

## A4 — Generate, grounded and cited

Two things to notice in the prompt below. The model is told to answer **only**
from the context, and it is given an explicit way out: `NOT IN CONTEXT`.

That escape hatch is a **generation-side** mitigation, and Part B uses it to
separate the two kinds of fault. Keep it in mind.


In [32]:
SYSTEM = (
    "You are answering questions about a codebase. "
    "Answer ONLY from the provided context. "
    "Cite the chunk id you used, in square brackets, after each claim. "
    "If the context does not contain the answer, reply exactly: NOT IN CONTEXT."
)


def build_context(retrieved):
    return "\n\n".join(
        f"--- chunk {ch['id']} ---\n{ch['text']}" for ch in retrieved
    )


def answer(query, k=None, system=SYSTEM, show_all=False):
    retrieved = retrieve(query, k=k, show_all=show_all)
    response = client.models.generate_content(
        model=config.CHAT_MODEL,
        contents=f"CONTEXT:\n{build_context(retrieved)}\n\nQUESTION: {query}",
        config=types.GenerateContentConfig(system_instruction=system),
    )
    print("\n" + "=" * 72)
    print(response.text)
    return response

In [33]:
_ = answer("How does login work?")

QUERY: How does login work?

   1. 0.7800  auth/login.py#L1-L9                  <-- retrieved
   2. 0.7571  auth/login.py#L14-L38                <-- retrieved
   3. 0.7234  auth/tokens.py#L1-L22                <-- retrieved
   4. 0.7189  docs/README.md#L1-L33                <-- retrieved

  margin between rank 1 and rank 2: 0.0229


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

---

# Part B — Break it

Five queries, five failure modes. For each one: run it, **read the score table
before you read the answer**, and decide whether what went wrong is a
**retrieval** fault or a **generation** fault.

You will be asked that question out loud, without notes, in the assessment.


### Q1 — bad chunking

`validate_token()` performs two independent checks: an expiry check and a
signature check. Find out which chunk each one landed in.


In [34]:
_ = answer("How is the token signature verified?", show_all=True)

QUERY: How is the token signature verified?

   1. 0.7530  auth/tokens.py#L43-L80               <-- retrieved
   2. 0.7483  auth/tokens.py#L1-L22                <-- retrieved
   3. 0.7138  auth/tokens.py#L83-L95               <-- retrieved
   4. 0.6850  auth/tokens.py#L36-L40               <-- retrieved
   5. 0.6735  auth/password.py#L1-L11              
   6. 0.6702  auth/login.py#L1-L9                  
   7. 0.6694  auth/tokens.py#L23-L24               
   8. 0.6682  config/settings.py#L1-L29            
   9. 0.6677  auth/login.py#L14-L38                
  10. 0.6633  auth/tokens.py#L27-L29               
  11. 0.6590  docs/ADR-002-token-expiry.md#L1-L44  
  12. 0.6587  auth/password.py#L21-L34             
  13. 0.6451  auth/login.py#L41-L50                
  14. 0.6417  auth/tokens.py#L32-L33               
  15. 0.6393  docs/README.md#L1-L33                
  16. 0.6103  billing/charges.py#L35-L49           
  17. 0.6053  auth/login.py#L53-L54                
  18. 0.6043  billi

### Q2 — missed context

Before you blame the model, check what was indexed.


In [35]:
_ = answer("How does the second factor get checked?", show_all=True)

QUERY: How does the second factor get checked?

   1. 0.7029  auth/password.py#L21-L34             <-- retrieved
   2. 0.7011  auth/login.py#L1-L9                  <-- retrieved
   3. 0.6929  auth/tokens.py#L1-L22                <-- retrieved
   4. 0.6895  config/settings.py#L1-L29            <-- retrieved
   5. 0.6864  billing/charges.py#L58-L60           
   6. 0.6858  auth/password.py#L1-L11              
   7. 0.6813  auth/tokens.py#L43-L80               
   8. 0.6753  auth/login.py#L53-L54                
   9. 0.6651  billing/charges.py#L35-L49           
  10. 0.6628  billing/charges.py#L1-L9             
  11. 0.6616  auth/login.py#L14-L38                
  12. 0.6616  docs/README.md#L1-L33                
  13. 0.6600  auth/tokens.py#L36-L40               
  14. 0.6596  auth/login.py#L41-L50                
  15. 0.6577  auth/login.py#L10-L11                
  16. 0.6576  docs/ADR-002-token-expiry.md#L1-L44  
  17. 0.6558  billing/refund.py#L1-L5              
  18. 0.6541  au

### Q3 — stale source

Three files in this corpus have an opinion about how long a token lasts. They do
not agree. Look at which one won, and then work out which one is *right* — and
be precise about whether the fault is a stale **index** or a stale **source**,
because they are different faults with different fixes.


In [37]:
_ = answer("How long does a session token last?", show_all=True)

QUERY: How long does a session token last?

   1. 0.7724  docs/ADR-002-token-expiry.md#L1-L44  <-- retrieved
   2. 0.7644  auth/tokens.py#L1-L22                <-- retrieved
   3. 0.7495  config/settings.py#L1-L29            <-- retrieved
   4. 0.7173  auth/login.py#L41-L50                <-- retrieved
   5. 0.7115  auth/tokens.py#L43-L80               
   6. 0.7040  auth/tokens.py#L83-L95               
   7. 0.6849  billing/charges.py#L35-L49           
   8. 0.6821  auth/login.py#L14-L38                
   9. 0.6742  billing/charges.py#L1-L9             
  10. 0.6729  docs/README.md#L1-L33                
  11. 0.6658  auth/login.py#L1-L9                  
  12. 0.6594  billing/charges.py#L58-L60           
  13. 0.6549  billing/charges.py#L52-L55           
  14. 0.6507  auth/login.py#L53-L54                
  15. 0.6407  auth/tokens.py#L36-L40               
  16. 0.6362  billing/refund.py#L1-L5              
  17. 0.6341  auth/password.py#L1-L11              
  18. 0.6331  auth/t

### Q4 — wrong retrieval

Look at the margin between rank 1 and rank 2 on this one. Then ask yourself what
a larger `k` would and would not have fixed.


In [38]:
_ = answer("What happens when a charge fails?", show_all=True)

QUERY: What happens when a charge fails?

   1. 0.7816  billing/refund.py#L28-L39            <-- retrieved
   2. 0.7810  billing/charges.py#L10-L11           <-- retrieved
   3. 0.7745  billing/refund.py#L1-L5              <-- retrieved
   4. 0.7638  billing/charges.py#L1-L9             <-- retrieved
   5. 0.7519  billing/charges.py#L14-L32           
   6. 0.7421  billing/charges.py#L35-L49           
   7. 0.7340  billing/refund.py#L10-L25            
   8. 0.7298  auth/login.py#L53-L54                
   9. 0.7140  billing/refund.py#L6-L7              
  10. 0.6930  billing/charges.py#L58-L60           
  11. 0.6831  config/settings.py#L1-L29            
  12. 0.6682  docs/README.md#L1-L33                
  13. 0.6675  auth/login.py#L10-L11                
  14. 0.6570  auth/login.py#L41-L50                
  15. 0.6558  auth/tokens.py#L23-L24               
  16. 0.6512  billing/charges.py#L52-L55           
  17. 0.6430  docs/ADR-002-token-expiry.md#L1-L44  
  18. 0.6401  auth/log

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

### Q5 — needle in a haystack

This question needs three facts that live in three different files. Retrieval
will find some of them.


In [28]:
_ = answer(
    "What happens to a user's active session when their subscription "
    "lapses mid-cycle?",
    show_all=True,
)

QUERY: What happens to a user's active session when their subscription lapses mid-cycle?

   1. 0.7404  docs/ADR-002-token-expiry.md#L1-L44  <-- retrieved
   2. 0.7118  billing/charges.py#L1-L60            <-- retrieved
   3. 0.6958  auth/login.py#L1-L54                 <-- retrieved
   4. 0.6924  config/settings.py#L1-L29            <-- retrieved
   5. 0.6924  auth/tokens.py#L1-L68                
   6. 0.6739  docs/README.md#L1-L33                
   7. 0.6576  billing/refund.py#L1-L39             
   8. 0.6429  auth/tokens.py#L61-L95               
   9. 0.6098  auth/password.py#L1-L34              
  10. 0.6058  billing/invoice.py#L1-L22            

  margin between rank 1 and rank 2: 0.0286


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

---

## The two-run demonstration

Take **one** query that failed above and run the generation step twice: once with
the `NOT IN CONTEXT` escape hatch, once without it.

Same retrieval. Same chunks. Opposite behaviour. That contrast is the clearest
thirty seconds in this lab, and it is worth having on screen when the examiner
asks you to classify the fault.


In [29]:
NO_ESCAPE = (
    "You are answering questions about a codebase. "
    "Answer from the provided context. "
    "Cite the chunk id you used, in square brackets, after each claim."
)

QUERY = "How does the second factor get checked?"

print("### WITH the escape hatch " + "#" * 40)
_ = answer(QUERY, system=SYSTEM)

print("\n\n### WITHOUT the escape hatch " + "#" * 37)
_ = answer(QUERY, system=NO_ESCAPE)

### WITH the escape hatch ########################################
QUERY: How does the second factor get checked?

   1. 0.7077  auth/tokens.py#L1-L68                <-- retrieved
   2. 0.7004  auth/password.py#L1-L34              <-- retrieved
   3. 0.6895  config/settings.py#L1-L29            <-- retrieved
   4. 0.6865  auth/login.py#L1-L54                 <-- retrieved

  margin between rank 1 and rank 2: 0.0072


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

---

# Part D — One improvement

Pick one failure, name the mode, apply the fix, show the before-and-after scores.

Below is a working AST-aware chunker, which is the fix for mode 1. It splits
Python along syntactic boundaries so that a function is never cut in half. Run
it, re-embed, and re-run Q1.

Note what it costs: it only works for Python, it needs a parser per language,
and it will happily emit a chunk far larger than your token budget if someone
writes a 900-line function. Fixed-size chunking has none of those problems. That
trade-off is the answer to "why doesn't everyone just do this?"


In [30]:
import ast


def chunk_file_ast(path, rel, chunk_tokens, overlap_tokens):
    """Split Python on top-level definitions; fall back to fixed-size."""
    source = path.read_text()
    if path.suffix != ".py":
        return chunk_file(path, rel, chunk_tokens, overlap_tokens)

    try:
        tree = ast.parse(source)
    except SyntaxError:
        return chunk_file(path, rel, chunk_tokens, overlap_tokens)

    lines = source.splitlines()
    spans, cursor = [], 0
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            start = min(
                [d.lineno for d in node.decorator_list] + [node.lineno]
            ) - 1
            if start > cursor:
                spans.append((cursor, start))       # module-level preamble
            spans.append((start, node.end_lineno))
            cursor = node.end_lineno
    if cursor < len(lines):
        spans.append((cursor, len(lines)))

    out = []
    for start, end in spans:
        text = "\n".join(lines[start:end]).strip("\n")
        if not text.strip():
            continue
        out.append({
            "id": f"{rel}#L{start+1}-L{end}",
            "path": rel, "start": start + 1, "end": end,
            "text": text, "tokens": len(tokenize(text)),
        })
    return out


# Re-index with AST-aware chunking, then re-embed and re-run Q1.
import pathlib

_root = pathlib.Path(config.CORPUS_DIR)
_patterns = load_ragignore(_root)
chunks = []
for p in sorted(_root.rglob("*")):
    if not p.is_file() or p.suffix not in config.INDEXABLE_SUFFIXES:
        continue
    rel = str(p.relative_to(_root))
    if is_ignored(rel, _patterns) or p.name == "__init__.py":
        continue
    chunks += chunk_file_ast(p, rel, config.CHUNK_TOKENS, config.OVERLAP_TOKENS)

print(f"{len(chunks)} AST chunks\n")
for ch in chunks:
    print(f"  {ch['id']:40s} {ch['tokens']:4d} tokens")

doc_vectors = embed_documents([c["text"] for c in chunks])
print()
_ = answer("How is the token signature verified?", show_all=True)

31 AST chunks

  auth/login.py#L1-L9                        46 tokens
  auth/login.py#L10-L11                      20 tokens
  auth/login.py#L14-L38                     201 tokens
  auth/login.py#L41-L50                      93 tokens
  auth/login.py#L53-L54                      26 tokens
  auth/password.py#L1-L11                    26 tokens
  auth/password.py#L12-L18                   77 tokens
  auth/password.py#L21-L34                   97 tokens
  auth/tokens.py#L1-L22                     125 tokens
  auth/tokens.py#L23-L24                     24 tokens
  auth/tokens.py#L27-L29                     35 tokens
  auth/tokens.py#L32-L33                     32 tokens
  auth/tokens.py#L36-L40                     60 tokens
  auth/tokens.py#L43-L80                    285 tokens
  auth/tokens.py#L83-L95                    162 tokens
  billing/charges.py#L1-L9                   31 tokens
  billing/charges.py#L10-L11                 24 tokens
  billing/charges.py#L14-L32                167 to

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.6-flash\nPlease retry in 10.708867898s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'

---

## Before your assessment

You have five minutes and an examiner who will pick which of you presents.

- [ ] The notebook runs top to bottom without editing.
- [ ] A **score table** is visible on screen, not just an answer.
- [ ] One grounded answer cites the correct chunk.
- [ ] One Part B failure runs live and fails in public.
- [ ] Every member can answer, unscripted: *retrieval fault or generation fault,
      and what would fix it?*
- [ ] Outputs are saved in the committed notebook, as the examiner's fallback.
- [ ] **No API key anywhere in the repository.**

If your room's connection is unreliable, set `config.OFFLINE = True` and confirm
the whole notebook still runs. Only the generation cells need the network.
